# Index Implied Volatility vs Index Price Movements

Cross-reference index prices (`indicies.parquet`) with their implied volatility
(`indices_ivol.xlsx`) for all available indices.

| Index | Price Source | IVol Source | Notes |
|---|---|---|---|
| UKX | UKX | UKX | FTSE 100 |
| CAC | CAC | CAC | CAC 40 |
| DAX | DAX | DAX | DAX 40 |
| MIB | MIB | MIB | FTSE MIB |
| IBEX | IBEX | IBEX | IBEX 35 |
| SPX | SPX | SPX | S&P 500 (ivol ≈ VIX) |
| CCMP/NDX | CCMP (price) | NDX (ivol) | Nasdaq — Composite vs 100 |

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

import subprocess as _sp
PROJECT_ROOT = Path(_sp.check_output(
    "git rev-parse --show-toplevel", shell=True, text=True, cwd="."
).strip())

# --- Index prices from parquet ---
idx_prices_raw = pd.read_parquet(PROJECT_ROOT / 'equities' / 'indicies.parquet')
idx_prices = idx_prices_raw.pivot(index='date', columns='ticker', values='close').sort_index()

# --- Implied vol from Excel (one sheet per index) ---
ivol_xl = pd.ExcelFile(PROJECT_ROOT / 'data' / 'indices_ivol.xlsx')
ivol_sheets = {}
for sheet in ivol_xl.sheet_names:
    tmp = pd.read_excel(ivol_xl, sheet_name=sheet)
    tmp.columns = ['date', 'ivol']
    tmp['date'] = pd.to_datetime(tmp['date'], errors='coerce')
    tmp = tmp.dropna(subset=['date']).set_index('date').sort_index()
    ivol_sheets[sheet] = tmp['ivol']

ivol_df = pd.DataFrame(ivol_sheets)

# Mapping: price ticker -> ivol sheet (CCMP price ↔ NDX ivol)
TICKER_MAP = {
    'UKX': 'UKX', 'CAC': 'CAC', 'DAX': 'DAX',
    'MIB': 'MIB', 'IBEX': 'IBEX', 'SPX': 'SPX',
    'CCMP': 'NDX',
}

print(f"Index prices : {idx_prices.shape[1]} tickers, {idx_prices.index.min().date()} \u2192 {idx_prices.index.max().date()}")
print(f"Implied vol  : {ivol_df.shape[1]} tickers, {ivol_df.index.min().date()} \u2192 {ivol_df.index.max().date()}")
print(f"\nPrice tickers : {list(idx_prices.columns)}")
print(f"IVol tickers  : {list(ivol_df.columns)}")

In [ ]:
# Dashboard: Index Price vs Implied Vol for each index
# One subplot row per index: left = price + ivol overlay, right = 30-day rolling correlation

indices = list(TICKER_MAP.keys())
n = len(indices)

fig = make_subplots(
    rows=n, cols=2, shared_xaxes='columns',
    vertical_spacing=0.03, horizontal_spacing=0.08,
    subplot_titles=[item for idx in indices for item in (f'{idx} \u2014 Price vs IVol', f'{idx} \u2014 30d Rolling Correlation')],
    specs=[[{"secondary_y": True}, {}] for _ in indices],
)

colors_price = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336', '#00BCD4', '#795548']
colors_ivol  = ['#90CAF9', '#A5D6A7', '#FFE0B2', '#CE93D8', '#EF9A9A', '#80DEEA', '#BCAAA4']

for i, ticker in enumerate(indices, 1):
    ivol_key = TICKER_MAP[ticker]
    
    # Align to common dates
    if ticker not in idx_prices.columns or ivol_key not in ivol_df.columns:
        continue
    common = idx_prices[ticker].dropna().index.intersection(ivol_df[ivol_key].dropna().index)
    px_series = idx_prices.loc[common, ticker]
    iv_series = ivol_df.loc[common, ivol_key]
    
    # Left panel: price + ivol
    fig.add_trace(go.Scatter(x=common, y=px_series, name=f'{ticker} Price',
        line=dict(color=colors_price[i-1], width=1.2), showlegend=(i==1)),
        row=i, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=common, y=iv_series, name=f'{ivol_key} IVol',
        line=dict(color=colors_ivol[i-1], width=1, dash='dot'), showlegend=(i==1)),
        row=i, col=1, secondary_y=True)
    
    fig.update_yaxes(title_text='Price', row=i, col=1, secondary_y=False)
    fig.update_yaxes(title_text='IVol', row=i, col=1, secondary_y=True)
    
    # Right panel: 30-day rolling correlation between daily returns and ivol changes
    px_ret = px_series.pct_change()
    iv_chg = iv_series.pct_change()
    roll_corr = px_ret.rolling(30).corr(iv_chg)
    
    fig.add_trace(go.Scatter(x=common, y=roll_corr, name=f'{ticker} corr',
        line=dict(color=colors_price[i-1], width=1), showlegend=False),
        row=i, col=2)
    fig.add_hline(y=0, line_dash='dash', line_color='grey', row=i, col=2)
    fig.update_yaxes(title_text='Corr', range=[-1, 1], row=i, col=2)

fig.update_layout(
    height=350 * n, template='plotly_white',
    title='Index Prices vs Implied Volatility \u2014 All Indices',
    hovermode='x unified', showlegend=True,
)
fig.show()

In [ ]:
# Summary table: IVol statistics and latest values per index

summary_rows = []
for ticker in TICKER_MAP:
    ivol_key = TICKER_MAP[ticker]
    if ivol_key not in ivol_df.columns or ticker not in idx_prices.columns:
        continue
    iv = ivol_df[ivol_key].dropna()
    px = idx_prices[ticker].dropna()
    
    # Last overlapping dates
    common = px.index.intersection(iv.index)
    if len(common) == 0:
        continue
    
    last_date = common.max()
    px_ret_1y = px.pct_change(252).loc[:last_date].iloc[-1] * 100 if len(px) > 252 else np.nan
    
    summary_rows.append({
        'Index': ticker,
        'IVol Ticker': ivol_key,
        'Price Last Date': px.index.max().date(),
        'IVol Last Date': iv.index.max().date(),
        'IVol Latest': f"{iv.iloc[-1]:.1f}",
        'IVol Mean': f"{iv.mean():.1f}",
        'IVol Median': f"{iv.median():.1f}",
        'IVol Max': f"{iv.max():.1f}",
        'IVol Min': f"{iv.min():.1f}",
        'Price 1Y Ret%': f"{px_ret_1y:.1f}" if not np.isnan(px_ret_1y) else 'N/A',
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)